> Check individual signals according to the community they belong.

In [ ]:
import pathlib

import yaml
import pandas as pd

import plotly
from bokeh.io import output_notebook
from bokeh.resources import INLINE
from bokeh.plotting import show

from trapallada.analysis import SignalsSuperGraph, read_graph
from trapallada.plot import full_signal

import ipywidgets as widgets
from IPython.display import clear_output

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
output_notebook(resources=INLINE)

So that *plotly* pictures can be exported to HTML

In [ ]:
plotly.offline.init_notebook_mode()

# Input data

Particular *sub*directories with
- the trained model,
- the corresponding graph, and
- the data.

They should be a `str` so that parameters can be injected with papermill.

In [ ]:
model_dir = '../results/cvae/min_count_rate_gti_with_background/fft_vanilla_z_5'
graph_dir = '../results/jesus/starwVanillaZ05_00/graphs/sigsum'

Turning the `str` `pathlib.Path`s

In [ ]:
model_dir = pathlib.Path(model_dir)

It is used to infer the *data directory*

In [ ]:
with (model_dir / 'extended_parameters.yaml').open() as f:
    parameters = yaml.safe_load(f)
data_dir = (pathlib.Path.home() / parameters['dataset']['files']['signals_file']).parent
display(data_dir)
assert data_dir.exists()

From the *results* directory, only one file is required

In [ ]:
analysis_file = pathlib.Path(model_dir) / 'analysis.parquet'

The graphs (nodes and edges) is read.

In [ ]:
nodes_df, edges_df = read_graph(graph_dir)

A `DataFrame` *merging* data from the *analysis* with that from the graph.

In [ ]:
analysis_cols = ['signal', 'main_type', 'main_id', 'RA', 'DEC', 'CRA', 'CDEC']
nodes_cols = ['Id', 'leiden', 'x', 'y']
analysis_df = pd.read_parquet(analysis_file)
analyzed_signals_df =  analysis_df[analysis_cols].drop_duplicates().merge(
    nodes_df[nodes_cols], left_on='signal', right_on='Id', how='left'
)
analyzed_signals_df = analyzed_signals_df[analysis_cols + nodes_cols[1:]]
analyzed_signals_df.head()

In [ ]:
graph_plot = SignalsSuperGraph(nodes_df, edges_df, data_dir)
fig = graph_plot.make_fig()
show(fig)

In [ ]:
sorted_signals = analyzed_signals_df['signal'].sort_values()

signal_dropdown = widgets.Dropdown(
    options=sorted_signals,
    value=sorted_signals.iloc[0],
    description='Signal:',
    disabled=False
)

def plot_signal(change):
    
    clear_output(wait=True)
    selected_signal = change['new']
    fig = full_signal(selected_signal, pathlib.Path(data_dir) / 'full_signals')
    display(signal_dropdown)
    display(fig)
    display(analyzed_signals_df[analyzed_signals_df['signal'] == selected_signal])

signal_dropdown.observe(plot_signal, names='value')
display(signal_dropdown)